# 01 · Build the (image → JSON) dataset from rendered table images

This notebook materializes the **synthetic complex-table corpus** (Donut-style:
pixel-perfect labels by construction, since the JSON and the image come from the
same `TableSpec`). Each exposed utility is demonstrated **individually** first,
then the full corpus is built from `configs/config.yaml`.

**Why synthetic?** No Hugging Face datasets allowed + the new vision pathway is
random-init → it needs thousands of perfectly labelled pairs. Swap in real
annotated tables later by writing the same JSONL manifest schema.

In [ ]:
# --- bootstrap: make the src/ package importable from notebooks/ ---
import sys, os
from pathlib import Path
REPO = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)  # so relative paths in configs/config.yaml resolve

from gemma_ft_json.config import load_config
cfg = load_config("configs/config.yaml")
print("config loaded; data_root =", cfg.paths.data_root)


## 1. Exposed utility — sample one logical `TableSpec`

In [ ]:
from gemma_ft_json.data import SyntheticTableGenerator
from gemma_ft_json.utils import set_seed

set_seed(cfg.dataset.seed)
gen = SyntheticTableGenerator(cfg.dataset)
spec = gen.sample_spec()
print("headers      :", spec.headers)
print("header_groups:", spec.header_groups)
print("row_spans    :", spec.row_spans)
print("style        :", spec.style)
spec.rows[:3]

## 2. Exposed utility — render the spec to an image

In [ ]:
import matplotlib.pyplot as plt
img = gen.render(spec, cfg.dataset.image_size)
plt.figure(figsize=(6, 6)); plt.imshow(img); plt.axis("off")
plt.title("rendered table (model input)"); plt.show()

## 3. Exposed utilities — the three aligned curriculum targets

In [ ]:
import json
print("READ target      :", SyntheticTableGenerator.to_read_text(spec)[:160], "...")
print()
print("LINEARIZE target :")
print(SyntheticTableGenerator.to_linearized(spec)[:300])
print()
print("JSON target      :")
print(json.dumps(SyntheticTableGenerator.to_json_obj(spec), indent=2)[:500])

## 4. Build the full train/val corpus (paths & sizes from config)

In [ ]:
from tqdm.auto import tqdm
from gemma_ft_json.data import build_dataset

for split, n, manifest, off in (
    ("train", cfg.dataset.num_train_samples, cfg.paths.manifest_train, 0),
    ("val",   cfg.dataset.num_val_samples,   cfg.paths.manifest_val,   10_000),
):
    bar = tqdm(total=n, desc=f"render {split}")
    wrote = build_dataset(cfg.dataset, cfg.paths.raw_images_dir, manifest, n,
                          split, seed_offset=off, progress_cb=lambda i, t: bar.update(1))
    bar.close()
    print(f"{split}: {wrote} samples -> {manifest}")

## 5. Sanity: read the manifest back

In [ ]:
with open(cfg.paths.manifest_train) as fh:
    first = json.loads(fh.readline())
print({k: (v[:80] + '...') if isinstance(v, str) and len(v) > 80 else v
       for k, v in first.items()})